# ========================================
# VENDOR ANALYSIS PROJECT - DATABASE LOADING
# Notebook 3: Load Scraped Data into Database
# ========================================

In [1]:
# CELL 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Float, Date, Boolean, DateTime, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import os
import glob

print("✅ Setup complete\n")

Mounted at /content/drive
✅ Setup complete



In [2]:
# CELL 2: Define Enhanced Database Schema (Based on Your Bedrock Data)
Base = declarative_base()

class Vendor(Base):
    """Enhanced Vendor table based on Bedrock scraped data"""
    __tablename__ = 'vendors'

    # Primary Key
    id = Column(Integer, primary_key=True, autoincrement=True)

    # Basic Information
    bedrock_id = Column(String(50), unique=True, index=True)
    vendor_number = Column(String(50), index=True)
    vendor_name = Column(String(255), index=True)
    dba_name = Column(String(255))

    # Status Information
    onboarding_status = Column(String(100))
    audit_status = Column(String(100))
    login_status = Column(String(50))

    # Contact Information
    primary_contact = Column(String(255))
    primary_email = Column(String(255))
    phone = Column(String(50))
    primary_contact_first_name = Column(String(100))
    primary_contact_last_name = Column(String(100))
    primary_contact_phone = Column(String(50))
    primary_contact_email = Column(String(255))

    # Financial Information (Critical for Analysis)
    tax_id = Column(String(50))
    total_spend = Column(Float, default=0.0)
    spend_2024 = Column(Float, default=0.0)
    spend_2023 = Column(Float, default=0.0)
    spend_2022 = Column(Float, default=0.0)
    estimated_annual_spend = Column(Float)

    # Vendor Details
    vendor_type = Column(String(100))
    payment_terms = Column(String(100))
    one_time_vendor = Column(String(10))
    audit_period = Column(String(50))

    # Company Information
    registered_company_name = Column(String(255))
    company_dba_name = Column(String(255))
    business_entity_type = Column(String(100))
    year_founded = Column(String(10))
    duns_number = Column(String(50))

    # Address Information
    registered_address_1 = Column(Text)
    registered_address_2 = Column(Text)
    city = Column(String(100))
    county = Column(String(100))
    state = Column(String(50), index=True)
    postal_code = Column(String(20))
    country = Column(String(100))

    # Additional Information
    website_url = Column(String(255))
    compliance_verification = Column(String(50))
    banking_verification = Column(String(50))
    legacy_vendor_id = Column(String(50))

    # Audit Trail
    modified_by = Column(String(100))
    modified_date = Column(DateTime)
    modified_type = Column(String(50))
    created_by = Column(String(100))
    scraped_at = Column(DateTime)

    def __repr__(self):
        return f"<Vendor(bedrock_id='{self.bedrock_id}', name='{self.vendor_name}')>"

print("✅ Database schema defined\n")

✅ Database schema defined



/tmp/ipython-input-56866202.py:2: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [3]:
# CELL 3: Create/Connect to Existing Database
DATABASE_PATH = '/content/drive/MyDrive/vendor-analysis-project/vendors.db'

# Check if database already exists
db_exists = os.path.exists(DATABASE_PATH)

engine = create_engine(f'sqlite:///{DATABASE_PATH}', echo=False)
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine)
session = Session()

if db_exists:
    print(f"✅ Connected to existing database: {DATABASE_PATH}")
else:
    print(f"✅ Created new database: {DATABASE_PATH}")

# Check existing records
existing_count = session.query(Vendor).count()
print(f"📊 Existing vendors in database: {existing_count}\n")

✅ Connected to existing database: /content/drive/MyDrive/vendor-analysis-project/vendors.db
📊 Existing vendors in database: 116



In [4]:
# CELL 4: Load Your Scraped CSV Data
DATA_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/raw'

# Find all CSV files
csv_files = glob.glob(f'{DATA_DIR}/vendors_*.csv')

print(f"Found {len(csv_files)} CSV files:\n")
for file in csv_files:
    file_size = os.path.getsize(file) / (1024 * 1024)  # MB
    print(f"  📄 {os.path.basename(file)} ({file_size:.2f} MB)")

# Use the complete file
complete_files = [f for f in csv_files if 'complete' in f]

if complete_files:
    latest_csv = max(complete_files, key=os.path.getctime)
    print(f"\n✅ Using: {os.path.basename(latest_csv)}\n")
else:
    # Use the most recent scraped file
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"\n✅ Using: {os.path.basename(latest_csv)}\n")

# Load CSV
print("Loading data...")
df = pd.read_csv(latest_csv, low_memory=False)
print(f"✅ Loaded {len(df)} vendors from CSV\n")

print("📋 Columns in dataset:")
print(df.columns.tolist())

print("\n📊 First 3 rows preview:")
print(df.head(3).to_string())

Found 35 CSV files:

  📄 vendors_scraped_20251013_091344.csv (0.00 MB)
  📄 vendors_scraped_20251013_125912.csv (0.00 MB)
  📄 vendors_scraped_20251013_130816.csv (0.00 MB)
  📄 vendors_scraped_20251013_132143.csv (0.00 MB)
  📄 vendors_scraped_20251013_144548.csv (0.00 MB)
  📄 vendors_scraped_20251015_070917.csv (0.00 MB)
  📄 vendors_complete_20251015_083900.csv (0.03 MB)
  📄 vendors_complete_20251022_141049.csv (0.03 MB)
  📄 vendors_complete_20251022_144946.csv (0.03 MB)
  📄 vendors_progress_10pages_20251029_070635.csv (0.24 MB)
  📄 vendors_progress_20pages_20251029_071338.csv (0.50 MB)
  📄 vendors_progress_30pages_20251029_072021.csv (0.77 MB)
  📄 vendors_progress_40pages_20251029_072658.csv (1.03 MB)
  📄 vendors_all_20251029_072743.csv (1.06 MB)
  📄 vendors_complete_all_20251029_073224.csv (0.00 MB)
  📄 vendors_complete_20251029_090814.csv (0.03 MB)
  📄 vendors_progress_50vendors_20251029_101835.csv (0.02 MB)
  📄 vendors_complete_with_details_20251029_103717.csv (0.02 MB)
  📄 vendors_c

In [5]:
# CELL 5: Data Cleaning & Preprocessing
print("\n" + "="*100)
print("DATA CLEANING & PREPROCESSING")
print("="*100 + "\n")

def clean_currency(value):
    """Convert currency string to float"""
    if pd.isna(value) or value == '' or value == 'nan':
        return 0.0
    try:
        cleaned = str(value).replace('$', '').replace(',', '').replace('USD', '').strip()
        return float(cleaned) if cleaned and cleaned != 'nan' else 0.0
    except:
        return 0.0

def clean_date(value):
    """Convert date string to datetime"""
    if pd.isna(value) or value == '' or value == 'nan':
        return None
    try:
        return pd.to_datetime(value)
    except:
        return None

def clean_text(value):
    """Clean text fields"""
    if pd.isna(value) or value == 'nan':
        return ''
    return str(value).strip()

# Clean financial columns
print("Cleaning financial data...")
financial_cols = ['total_spend', 'spend_2024', 'spend_2023', 'spend_2022', 'estimated_annual_spend']
for col in financial_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_currency)
        print(f"  ✓ Cleaned {col}: Min=${df[col].min():,.2f}, Max=${df[col].max():,.2f}")

# Clean date columns
print("\nCleaning date fields...")
date_cols = ['modified_date', 'scraped_at']
for col in date_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_date)
        print(f"  ✓ Cleaned {col}")

# Clean text columns
print("\nCleaning text fields...")
text_cols = ['vendor_name', 'dba_name', 'primary_contact', 'primary_email',
             'phone', 'city', 'state', 'vendor_type', 'payment_terms']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)
        non_empty = (df[col] != '').sum()
        print(f"  ✓ Cleaned {col}: {non_empty}/{len(df)} non-empty")

# Handle missing values
print("\nHandling missing values...")
df = df.fillna('')

# Remove duplicate vendors (based on bedrock_id or vendor_number)
print("\nRemoving duplicates...")
original_count = len(df)
if 'bedrock_id' in df.columns:
    df = df.drop_duplicates(subset=['bedrock_id'], keep='first')
elif 'vendor_number' in df.columns:
    df = df.drop_duplicates(subset=['vendor_number'], keep='first')
else:
    df = df.drop_duplicates(subset=['vendor_name'], keep='first')

duplicates_removed = original_count - len(df)
print(f"  ✓ Removed {duplicates_removed} duplicate vendors")

print(f"\n✅ Data cleaning complete!")
print(f"   Final dataset: {len(df)} unique vendors\n")



DATA CLEANING & PREPROCESSING

Cleaning financial data...
  ✓ Cleaned total_spend: Min=$0.00, Max=$23,752,510.53
  ✓ Cleaned spend_2024: Min=$0.00, Max=$18,529,241.53
  ✓ Cleaned spend_2023: Min=$0.00, Max=$3,130,872.39
  ✓ Cleaned spend_2022: Min=$0.00, Max=$5,387,085.96
  ✓ Cleaned estimated_annual_spend: Min=$0.00, Max=$0.00

Cleaning date fields...
  ✓ Cleaned modified_date
  ✓ Cleaned scraped_at

Cleaning text fields...
  ✓ Cleaned vendor_name: 100/100 non-empty
  ✓ Cleaned dba_name: 16/100 non-empty
  ✓ Cleaned primary_contact: 97/100 non-empty
  ✓ Cleaned primary_email: 95/100 non-empty
  ✓ Cleaned phone: 88/100 non-empty
  ✓ Cleaned city: 19/100 non-empty
  ✓ Cleaned state: 15/100 non-empty
  ✓ Cleaned vendor_type: 29/100 non-empty
  ✓ Cleaned payment_terms: 29/100 non-empty

Handling missing values...

Removing duplicates...
  ✓ Removed 0 duplicate vendors

✅ Data cleaning complete!
   Final dataset: 100 unique vendors



In [6]:
# CELL 6: Calculate Derived Metrics & Analytics Fields
print("="*100)
print("CALCULATING DERIVED METRICS")
print("="*100 + "\n")

# 1. Success indicators
df['has_spend'] = (df['total_spend'] > 0).astype(int)
df['has_contact_info'] = ((df['primary_email'] != '') | (df['phone'] != '')).astype(int)
df['is_active'] = df['login_status'].str.contains('Active|Registered', case=False, na=False).astype(int)

print("✓ Success indicators calculated")

# 2. Spend growth rate (2023 to 2024)
df['spend_growth_2023_2024'] = 0.0
mask = (df['spend_2023'] > 0)
df.loc[mask, 'spend_growth_2023_2024'] = (
    (df.loc[mask, 'spend_2024'] - df.loc[mask, 'spend_2023']) / df.loc[mask, 'spend_2023'] * 100
)
df['spend_growth_2023_2024'] = df['spend_growth_2023_2024'].replace([np.inf, -np.inf], 0).fillna(0)

print("✓ Spend growth rates calculated")

# 3. Vendor tier classification
def categorize_vendor_tier(spend):
    if spend >= 1000000:
        return 'Tier 1 - Premium (>$1M)'
    elif spend >= 100000:
        return 'Tier 2 - High Value ($100K-$1M)'
    elif spend >= 10000:
        return 'Tier 3 - Medium Value ($10K-$100K)'
    elif spend > 0:
        return 'Tier 4 - Low Value (<$10K)'
    else:
        return 'Tier 5 - No Spend'

df['vendor_tier'] = df['total_spend'].apply(categorize_vendor_tier)

print("✓ Vendor tiers calculated")

# 4. Regional grouping
if 'state' in df.columns and not df['state'].isna().all():
    df['region'] = df['state'].replace('', 'Unknown')

    # US Regional grouping
    northeast = ['CT', 'ME', 'MA', 'NH', 'RI', 'VT', 'NJ', 'NY', 'PA']
    southeast = ['DE', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'WV', 'AL', 'KY', 'MS', 'TN', 'AR', 'LA', 'OK', 'TX']
    midwest = ['IL', 'IN', 'MI', 'OH', 'WI', 'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD']
    west = ['AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY', 'AK', 'CA', 'HI', 'OR', 'WA']

    df['us_region'] = 'Other'
    df.loc[df['state'].isin(northeast), 'us_region'] = 'Northeast'
    df.loc[df['state'].isin(southeast), 'us_region'] = 'Southeast'
    df.loc[df['state'].isin(midwest), 'us_region'] = 'Midwest'
    df.loc[df['state'].isin(west), 'us_region'] = 'West'
else:
    df['region'] = 'Unknown'
    df['us_region'] = 'Unknown'

print("✓ Regional classifications calculated")

# 5. Activity score (0-100)
df['activity_score'] = (
    (df['is_active'] * 25) +
    (df['has_spend'] * 25) +
    ((df['primary_email'] != '').astype(int) * 25) +
    ((df['phone'] != '').astype(int) * 25)
)

print("✓ Activity scores calculated")

# 6. Risk classification
def classify_risk(growth_rate, spend):
    if spend == 0:
        return 'No Activity'
    elif growth_rate < -30:
        return 'High Risk'
    elif growth_rate < -10:
        return 'Medium Risk'
    elif growth_rate < 0:
        return 'Low Risk'
    else:
        return 'Growing/Stable'

df['risk_category'] = df.apply(lambda x: classify_risk(x['spend_growth_2023_2024'], x['total_spend']), axis=1)

print("✓ Risk categories calculated")

# 7. Engagement level
def classify_engagement(score):
    if score >= 75:
        return 'Highly Engaged'
    elif score >= 50:
        return 'Moderately Engaged'
    elif score >= 25:
        return 'Low Engagement'
    else:
        return 'No Engagement'

df['engagement_level'] = df['activity_score'].apply(classify_engagement)

print("✓ Engagement levels calculated")

print(f"\n✅ All metrics calculated!\n")

# Show summary statistics
print("📊 METRICS SUMMARY:")
print("="*100)
print(f"\nVendor Tier Distribution:")
print(df['vendor_tier'].value_counts())
print(f"\nRisk Category Distribution:")
print(df['risk_category'].value_counts())
print(f"\nEngagement Level Distribution:")
print(df['engagement_level'].value_counts())
print(f"\nUS Region Distribution:")
print(df['us_region'].value_counts())


CALCULATING DERIVED METRICS

✓ Success indicators calculated
✓ Spend growth rates calculated
✓ Vendor tiers calculated
✓ Regional classifications calculated
✓ Activity scores calculated
✓ Risk categories calculated
✓ Engagement levels calculated

✅ All metrics calculated!

📊 METRICS SUMMARY:

Vendor Tier Distribution:
vendor_tier
Tier 5 - No Spend                     58
Tier 2 - High Value ($100K-$1M)       13
Tier 4 - Low Value (<$10K)            12
Tier 1 - Premium (>$1M)               11
Tier 3 - Medium Value ($10K-$100K)     6
Name: count, dtype: int64

Risk Category Distribution:
risk_category
No Activity       58
High Risk         22
Growing/Stable    18
Medium Risk        2
Name: count, dtype: int64

Engagement Level Distribution:
engagement_level
Highly Engaged        90
Moderately Engaged    10
Name: count, dtype: int64

US Region Distribution:
us_region
Other    100
Name: count, dtype: int64


In [7]:
# CELL 7: Insert Data into Database
print("\n" + "="*100)
print("LOADING DATA INTO DATABASE")
print("="*100 + "\n")

insert_count = 0
update_count = 0
error_count = 0
skipped_count = 0

print(f"Processing {len(df)} vendors...")

for idx, row in df.iterrows():
    try:
        # Check if vendor already exists
        if 'bedrock_id' in row and row['bedrock_id'] != '':
            existing = session.query(Vendor).filter_by(bedrock_id=row['bedrock_id']).first()
        elif 'vendor_number' in row and row['vendor_number'] != '':
            existing = session.query(Vendor).filter_by(vendor_number=row['vendor_number']).first()
        else:
            existing = None

        if existing:
            skipped_count += 1
            if skipped_count % 50 == 0:
                print(f"  ⏭  Skipped {skipped_count} existing vendors...")
            continue

        # Create vendor object
        vendor = Vendor(
            bedrock_id=row.get('bedrock_id', ''),
            vendor_number=row.get('vendor_number', ''),
            vendor_name=row.get('vendor_name', ''),
            dba_name=row.get('dba_name', ''),
            onboarding_status=row.get('onboarding_status', ''),
            audit_status=row.get('audit_status', ''),
            login_status=row.get('login_status', ''),
            primary_contact=row.get('primary_contact', ''),
            primary_email=row.get('primary_email', ''),
            phone=row.get('phone', ''),
            tax_id=row.get('tax_id', ''),
            total_spend=float(row.get('total_spend', 0.0)),
            spend_2024=float(row.get('spend_2024', 0.0)),
            spend_2023=float(row.get('spend_2023', 0.0)),
            spend_2022=float(row.get('spend_2022', 0.0)),
            vendor_type=row.get('vendor_type', ''),
            payment_terms=row.get('payment_terms', ''),
            one_time_vendor=row.get('one_time_vendor', ''),
            audit_period=row.get('audit_period', ''),
            registered_company_name=row.get('registered_company_name', ''),
            company_dba_name=row.get('company_dba_name', ''),
            registered_address_1=row.get('registered_address_1', ''),
            registered_address_2=row.get('registered_address_2', ''),
            city=row.get('city', ''),
            county=row.get('county', ''),
            state=row.get('state', ''),
            postal_code=row.get('postal_code', ''),
            country=row.get('country', ''),
            website_url=row.get('website_url', ''),
            duns_number=row.get('duns_number', ''),
            compliance_verification=row.get('compliance_verification', ''),
            banking_verification=row.get('banking_verification', ''),
            legacy_vendor_id=row.get('legacy_vendor_id', ''),
            modified_by=row.get('modified_by', ''),
            modified_date=row.get('modified_date') if pd.notna(row.get('modified_date')) else None,
            modified_type=row.get('modified_type', ''),
            created_by=row.get('created_by', ''),
            scraped_at=row.get('scraped_at') if pd.notna(row.get('scraped_at')) else datetime.now()
        )

        session.add(vendor)
        insert_count += 1

        # Commit every 100 vendors
        if insert_count % 100 == 0:
            session.commit()
            print(f"  ✓ Inserted {insert_count} vendors...")

    except Exception as e:
        error_count += 1
        if error_count <= 5:  # Only show first 5 errors
            print(f"  ✗ Error at row {idx}: {str(e)[:100]}")
        continue

# Final commit
try:
    session.commit()
    print(f"  ✓ Final commit successful")
except Exception as e:
    print(f"  ✗ Final commit error: {str(e)}")

print(f"\n✅ Database loading complete!")
print(f"   New vendors inserted: {insert_count}")
print(f"   Existing vendors skipped: {skipped_count}")
print(f"   Errors: {error_count}")



LOADING DATA INTO DATABASE

Processing 100 vendors...
  ⏭  Skipped 50 existing vendors...
  ✓ Final commit successful

✅ Database loading complete!
   New vendors inserted: 2
   Existing vendors skipped: 98
   Errors: 0


In [8]:
# CELL 8: Verify Database & Generate Statistics
print("\n" + "="*100)
print("DATABASE VERIFICATION & STATISTICS")
print("="*100 + "\n")

# Count total vendors
total_vendors = session.query(Vendor).count()
print(f"📊 Total vendors in database: {total_vendors}\n")

# Get sample vendors
print("Sample Vendors:")
sample_vendors = session.query(Vendor).limit(5).all()
for v in sample_vendors:
    print(f"  • {v.vendor_name[:50]:<50} | ID: {v.bedrock_id:<10} | Spend: ${v.total_spend:>12,.2f}")

# Calculate statistics
from sqlalchemy import func

stats = {
    'total_vendors': session.query(func.count(Vendor.id)).scalar(),
    'vendors_with_email': session.query(func.count(Vendor.id)).filter(Vendor.primary_email != '').scalar(),
    'vendors_with_phone': session.query(func.count(Vendor.id)).filter(Vendor.phone != '').scalar(),
    'vendors_with_spend': session.query(func.count(Vendor.id)).filter(Vendor.total_spend > 0).scalar(),
    'total_spend': session.query(func.sum(Vendor.total_spend)).scalar() or 0,
    'avg_spend': session.query(func.avg(Vendor.total_spend)).scalar() or 0,
    'max_spend': session.query(func.max(Vendor.total_spend)).scalar() or 0,
    'spend_2024': session.query(func.sum(Vendor.spend_2024)).scalar() or 0,
    'spend_2023': session.query(func.sum(Vendor.spend_2023)).scalar() or 0,
}

print(f"\n" + "="*100)
print("📈 DATABASE STATISTICS")
print("="*100)
print(f"\n{'Metric':<40} {'Value':>20}")
print("-" * 100)
print(f"{'Total Vendors':<40} {stats['total_vendors']:>20,}")
print(f"{'Vendors with Email':<40} {stats['vendors_with_email']:>20,}")
print(f"{'Vendors with Phone':<40} {stats['vendors_with_phone']:>20,}")
print(f"{'Vendors with Spend':<40} {stats['vendors_with_spend']:>20,}")
print(f"{'Total Spend (All Years)':<40} ${stats['total_spend']:>19,.2f}")
print(f"{'Average Spend per Vendor':<40} ${stats['avg_spend']:>19,.2f}")
print(f"{'Maximum Vendor Spend':<40} ${stats['max_spend']:>19,.2f}")
print(f"{'Total Spend 2024':<40} ${stats['spend_2024']:>19,.2f}")
print(f"{'Total Spend 2023':<40} ${stats['spend_2023']:>19,.2f}")



DATABASE VERIFICATION & STATISTICS

📊 Total vendors in database: 118

Sample Vendors:
  • October 13                                         | ID: 100423646  | Spend: $        0.00
  • STOR -IT SELF STORAGE                              | ID: 100046451  | Spend: $   23,799.02
  • Smart Building Technologies, LLC                   | ID: 100159577  | Spend: $        0.00
  • KRAUSE BROTHER'S CONSTRUCTION                      | ID: 100077182  | Spend: $      531.05
  • Pacific NW Bio WA LLC                              | ID: 100157152  | Spend: $        0.00

📈 DATABASE STATISTICS

Metric                                                  Value
----------------------------------------------------------------------------------------------------
Total Vendors                                             118
Vendors with Email                                        114
Vendors with Phone                                        107
Vendors with Spend                                         50
Tot

In [9]:
# CELL 9: Export Processed Data for PySpark
print("\n" + "="*100)
print("EXPORTING PROCESSED DATA FOR PYSPARK ANALYSIS")
print("="*100 + "\n")

# Add calculated metrics to export
export_df = df.copy()

# Export to processed folder
PROCESSED_DIR = '/content/drive/MyDrive/vendor-analysis-project/data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

processed_file = f'{PROCESSED_DIR}/vendors_processed_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df.to_csv(processed_file, index=False)

print(f"✅ Processed data exported to:")
print(f"   {processed_file}")
print(f"   Records: {len(export_df):,}")
print(f"   Columns: {len(export_df.columns)}")

print(f"\n📋 Exported Columns:")
for i, col in enumerate(export_df.columns, 1):
    print(f"   {i:2d}. {col}")

# Close session
session.close()

print("\n" + "="*100)
print("✅ DATABASE LOADING & PROCESSING COMPLETE!")
print("="*100)
print("\n📝 Next Steps:")
print("   1. ✅ Data scraped from website")
print("   2. ✅ Data loaded into SQLite database")
print("   3. ✅ Metrics calculated and exported")
print("   4. ➡️  Next: Open Notebook 4 for PySpark Big Data Processing")
print("="*100 + "\n")


EXPORTING PROCESSED DATA FOR PYSPARK ANALYSIS

✅ Processed data exported to:
   /content/drive/MyDrive/vendor-analysis-project/data/processed/vendors_processed_20251105_064503.csv
   Records: 100
   Columns: 55

📋 Exported Columns:
    1. bedrock_id
    2. vendor_number
    3. vendor_name
    4. dba_name
    5. onboarding_status
    6. audit_status
    7. primary_contact
    8. primary_email
    9. tax_id
   10. audit_period
   11. phone
   12. total_spend
   13. spend_2024
   14. spend_2023
   15. spend_2022
   16. legacy_vendor_id
   17. login_status
   18. modified_by
   19. modified_date
   20. modified_type
   21. created_by
   22. vendor_type
   23. payment_terms
   24. estimated_annual_spend
   25. one_time_vendor
   26. registered_company_name
   27. company_dba_name
   28. registered_address_1
   29. registered_address_2
   30. city
   31. county
   32. state
   33. postal_code
   34. country
   35. business_entity_type
   36. year_founded
   37. duns_number
   38. primary_co

In [10]:
# NEW CELL: Prepare data for export
# The 'is_deleted' column is not available in this dataset,
# so we will export all processed vendors.
export_df = df.copy()

print(f"✅ Prepared {len(export_df)} vendors for export")

# Export only active vendors
# export_df = df_active.copy() # This line is no longer needed

✅ Prepared 100 vendors for export
